# Amplitude reconstruction and CDF-like output

This notebook starts from the artifact created by notebook 01, runs the three registered finalist models, exports one CDF-like file per model, and compares their structure with the matching NASA reference.

**Before running:** finish notebook 01 first, then open this notebook from the repository root.

**You should finish with:** three model predictions and three CDF-like exports plotted on the calibrated frequency and virtual-height axes. The comparison checks compatibility; predicted amplitudes are not expected to equal measured NASA amplitudes.

In [ ]:
from pathlib import Path
import sys
import json
import matplotlib
_interactive = False
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
    _interactive = 'agg' not in str(matplotlib.get_backend()).lower()
except NameError:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
def show_plot():
    if _interactive:
        plt.show()
    plt.close()

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / 'pyproject.toml').is_file():
        ROOT = candidate
        break
else:
    raise RuntimeError('Open this notebook from inside the FINAL ISIS repository')
sys.path[:0] = [str(ROOT), str(ROOT / 'src')]
artifact_path = ROOT / 'outputs/notebooks/01_scan.npz'
model_config_path = ROOT / 'configs/model_candidates.json'
reference_cdf_path = ROOT / 'data/samples/i2_av_ksh_1972322002235_v01.cdf'
assert artifact_path.is_file(), 'Run notebook 01 first'
assert model_config_path.is_file()
assert reference_cdf_path.is_file()

from isis_research import ionogram
from isis_research.nasa.cdf_compare import compare_cdf_content
import importlib
from scripts.pipeline import infer_isis_model
infer_isis_model = importlib.reload(infer_isis_model)
candidate_checkpoint = infer_isis_model.candidate_checkpoint
infer = infer_isis_model.infer
load_model_candidates = infer_isis_model.load_model_candidates
from scripts.pipeline.export_model_as_cdf import _pair_name_from_scan
from isis_research.nasa.model_cdf import export_model_cdf, header_from_csa

scan = ionogram.read_validated(artifact_path)
print(scan.intensity.shape, scan.frequency_mhz[[0, -1]], scan.virtual_height_km[[0, -1]])
signal = 1.0 - scan.intensity
plt.figure(figsize=(12, 7))
plt.imshow(
    signal,
    cmap='magma',
    aspect='auto',
    origin='upper',
    extent=[float(scan.frequency_mhz[0]), float(scan.frequency_mhz[-1]), float(scan.virtual_height_km[-1]), float(scan.virtual_height_km[0])],
)
plt.title('Signal-positive calibrated CSA input')
plt.xlabel('frequency (MHz)')
plt.ylabel('virtual height (km)')
show_plot()

## Model prediction

The model receives the calibrated film signal and writes a prediction on the same axes. The validity mask is carried forward and is not treated as measured amplitude.

In [ ]:
candidate_config = load_model_candidates(model_config_path)
prediction_paths = {}
predictions = {}
for model_name in candidate_config['models']:
    checkpoint = candidate_checkpoint(model_name, model_config_path)
    prediction_path = ROOT / f'outputs/notebooks/01_prediction_{model_name}.npz'
    predictions[model_name] = infer(
        artifact_path,
        checkpoint,
        prediction_path,
        model_name=model_name,
        model_config=model_config_path,
    )
    prediction_paths[model_name] = prediction_path

fig, axes = plt.subplots(1, len(predictions), figsize=(18, 6), constrained_layout=True)
for axis, (model_name, prediction) in zip(axes, predictions.items()):
    axis.imshow(
        prediction,
        cmap='magma',
        aspect='auto',
        vmin=0,
        vmax=1,
        origin='upper',
        extent=[float(scan.frequency_mhz[0]), float(scan.frequency_mhz[-1]), float(scan.virtual_height_km[-1]), float(scan.virtual_height_km[0])],
    )
    axis.set_title(candidate_config['models'][model_name]['label'])
    axis.set_xlabel('frequency (MHz)')
    axis.set_ylabel('virtual height (km)')
fig.suptitle('Three calibrated Phase 6 finalist predictions')
show_plot()

## CDF-like export and reference comparison

Observation time comes from the verified pair name. Fields unavailable from the CSA image are written as explicit unknowns. The final comparison reports shared variables and any variables present in only one file.

In [ ]:
pair_name = _pair_name_from_scan(artifact_path, scan)
header = header_from_csa(pair_name, 'KSH', scan.frequency_mhz, scan.virtual_height_km)
for model_name, prediction_path in prediction_paths.items():
    cdf_path = ROOT / f'outputs/notebooks/01_model_{model_name}.cdf'
    values, provenance = export_model_cdf(prediction_path, header, cdf_path)
    comparison = compare_cdf_content(reference_cdf_path, cdf_path)
    print(json.dumps({
        'model': model_name,
        'output': str(cdf_path),
        'ampl_shape': list(values['ampl'].shape),
        'reference_cdf': reference_cdf_path.name,
        'shared_variable_count': len(comparison['shared_variables']),
        'only_in_reference': comparison['only_in_nasa'],
        'only_in_model': comparison['only_in_model'],
        'provenance': provenance,
    }, indent=2))